# Load Model

In [ ]:
import re
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel
from huggingface_hub import login
import time
import json
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

login(token=token)

MODEL_ID = "CohereLabs/tiny-aya-base"
ADAPTER_REPO = "legesher/language-decoded-lora-condition-2-ur-5k"
MAX_SEQ_LENGTH = 1024
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_model():
    dtype = torch.float32
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_REPO,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=dtype,
        load_in_4bit=torch.cuda.is_available(),
        token=token,
    )
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"  # required for correct batch generation in decoder-only models
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return model, tokenizer

Load adapters

In [ ]:
model, tokenizer = load_model()

In [ ]:
import time

start = time.time()
from tqdm import tqdm

# ─────────────────────────────────────────────
# BATCHED GENERATION  (replaces generate_text)
# ─────────────────────────────────────────────
def generate_texts_batch(prompts: list[str], max_new_tokens: int = 80, batch_size: int = 32, desc: str = "") -> list[str]:
    all_outputs = []
    model_device = next(model.parameters()).device
    batches = range(0, len(prompts), batch_size)

    for i in tqdm(batches, desc=desc or "Generating", unit="batch"):
        batch = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for output in outputs:
            text = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
            all_outputs.append(text)

    return all_outputs

**Load datasets**

MGSM

In [ ]:
mgsm_zh = pd.read_csv("https://raw.githubusercontent.com/google-research-datasets/MGSM-Rev2/main/MGSM-Rev2/mgsm_zh.tsv", sep="\t", header=None)
mgsm_zh = mgsm_zh.rename(columns={0: "Question", 1: "Answer"})

mgsm_es = pd.read_csv("https://raw.githubusercontent.com/google-research-datasets/MGSM-Rev2/main/MGSM-Rev2/mgsm_es.tsv", sep="\t", header=None)
mgsm_es = mgsm_es.rename(columns={0: "Question", 1: "Answer"})

mgsm_ur_ds = load_dataset("large-traversaal/mgsm_urdu_final")
mgsm_ur = mgsm_ur_ds["test"].to_pandas()
mgsm_ur = mgsm_ur.drop(columns=["question", "answer", "urdu_answer", "equation_solution"])
mgsm_ur = mgsm_ur.rename(columns={"urdu_question": "Question", "answer_number": "Answer"})


XNLI

In [ ]:
xnli_zh = load_dataset("facebook/xnli", "zh")["test"].to_pandas()
xnli_es = load_dataset("facebook/xnli", "es")["test"].to_pandas()
xnli_ur = load_dataset("facebook/xnli", "ur")["test"].to_pandas()

CSQA

In [ ]:
csqa_es = load_dataset("INK-USC/xcsr", "X-CSQA-es")["validation"].to_pandas()
csqa_zh = load_dataset("INK-USC/xcsr", "X-CSQA-zh")["validation"].to_pandas()
csqa_ur = load_dataset("INK-USC/xcsr", "X-CSQA-ur")["validation"].to_pandas()

def prepare_csqa(df):
    df = df[["question", "answerKey"]].copy()
    df["stem"] = df["question"].apply(lambda x: x["stem"])
    df["choice_texts"] = df["question"].apply(lambda x: list(x["choices"]["text"]))
    df["A"] = df["choice_texts"].apply(lambda x: x[0])
    df["B"] = df["choice_texts"].apply(lambda x: x[1])
    df["C"] = df["choice_texts"].apply(lambda x: x[2])
    df["D"] = df["choice_texts"].apply(lambda x: x[3])
    df["E"] = df["choice_texts"].apply(lambda x: x[4])
    return df[["stem", "A", "B", "C", "D", "E", "answerKey"]]

csqa_es = prepare_csqa(csqa_es)
csqa_zh = prepare_csqa(csqa_zh)
csqa_ur = prepare_csqa(csqa_ur)

MGSM Functions

In [ ]:
def build_mgsm_prompt(question: str, lang: str) -> str:
    templates = {
        "en": f"""Solve the following math word problem.
Give the final answer as a number only.
Question: {question}
Answer:""",
        "es": f"""Resuelve el siguiente problema matemático.
Da la respuesta final solo como un número.
Pregunta: {question}
Respuesta:""",
        "zh": f"""请解答下面的数学应用题。
最终答案只输出数字。
题目: {question}
答案:""",
        "ur": f"""مندرجہ ذیل ریاضی کے سوال کو حل کریں۔
صرف عددی شکل میں آخری جواب دیں۔
سوال: {question}
جواب:"""
    }
    return templates[lang]

def extract_number(text: str):
    matches = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return matches[-1] if matches else None

def normalize_number_string(x):
    x = str(x).strip().replace(",", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", x)
    return matches[-1] if matches else None

def evaluate_mgsm(df: pd.DataFrame, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [build_mgsm_prompt(row["Question"], lang) for _, row in eval_df.iterrows()]
    outputs = generate_texts_batch(prompts, max_new_tokens=80, batch_size=batch_size)

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_number(output)
        gold = normalize_number_string(row["Answer"])
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "question": row["Question"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)

XNLI Functions

In [ ]:
LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

NATIVE_LABEL_MAP = {
    "蕴含": "entailment", "蕴涵": "entailment",
    "矛盾": "contradiction",
    "中立": "neutral",
    "implicación": "entailment", "implicacion": "entailment",
    "contradicción": "contradiction", "contradiccion": "contradiction",
    "لازمی": "entailment",
    "تردید": "contradiction",
    "غیرجانبدار": "neutral",
}

def build_xnli_prompt(row, lang: str) -> str:
    templates = {
        "en": f"""Read the premise and hypothesis below.
Decide whether the hypothesis is entailed by the premise, contradicted by the premise, or neither.
Reply with only one word: entailment, contradiction, or neutral.
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
Answer:""",
        "es": f"""Lee la premisa y la hipótesis a continuación.
Decide si la hipótesis se sigue de la premisa, la contradice, o no es ninguna de las dos.
Responde con solo una palabra: entailment, contradiction, o neutral.
Premisa: {row['premise']}
Hipótesis: {row['hypothesis']}
Respuesta:""",
        "zh": f"""请阅读下面的前提和假设。
判断假设是否被前提蕴含、与前提矛盾，或两者都不是。
只回答一个词：entailment、contradiction 或 neutral。
前提: {row['premise']}
假设: {row['hypothesis']}
答案:""",
        "ur": f"""نیچے دی گئی premise اور hypothesis کو پڑھیں۔
فیصلہ کریں کہ hypothesis، premise سے لازم آتی ہے، اس کی تردید کرتی ہے، یا دونوں میں سے کوئی نہیں۔
صرف ایک لفظ میں جواب دیں: entailment، contradiction، یا neutral۔
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
جواب:"""
    }
    return templates[lang]

def extract_xnli_label(text: str):
    text_lower = text.strip().lower()
    for label in ["entailment", "contradiction", "neutral"]:
        if re.search(rf"\b{label}\b", text_lower):
            return label
    for native, english in NATIVE_LABEL_MAP.items():
        if native in text.strip():
            return english
    return None

def normalize_xnli_gold(label):
    if isinstance(label, str):
        return label.strip().lower()
    return LABEL_MAP.get(int(label), None)

def evaluate_xnli(df: pd.DataFrame, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [build_xnli_prompt(row, lang) for _, row in eval_df.iterrows()]
    outputs = generate_texts_batch(prompts, max_new_tokens=10, batch_size=batch_size)

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_xnli_label(output)
        gold = normalize_xnli_gold(row["label"])
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "premise": row["premise"],
            "hypothesis": row["hypothesis"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)


CSQA Functions

In [ ]:
def build_csqa_prompt(row, lang: str) -> str:
    templates = {
        "en": f"""Choose the best answer to the following commonsense question.
Reply with only one letter: A, B, C, D, or E.
Question: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Answer:""",
        "es": f"""Elige la mejor respuesta para la siguiente pregunta de sentido común.
Responde solo con una letra: A, B, C, D o E.
Pregunta: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Respuesta:""",
        "zh": f"""请选择下面这个常识问题的最佳答案。
只回答一个字母：A、B、C、D 或 E。
问题: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
答案:""",
        "ur": f"""نیچے دیے گئے عمومی فہم کے سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف میں جواب دیں: A، B، C، D، یا E۔
سوال: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
جواب:"""
    }
    return templates[lang]

def extract_choice(text: str):
    text = text.strip().upper()
    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)
    return None

def evaluate_csqa(df: pd.DataFrame, lang: str, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [build_csqa_prompt(row, lang) for _, row in eval_df.iterrows()]
    outputs = generate_texts_batch(prompts, max_new_tokens=10, batch_size=batch_size)

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_choice(output)
        gold = str(row["answerKey"]).strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "stem": row["stem"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)

Evals

In [ ]:
def results_to_jsonable(results):
    jsonable = {"summary": results["summary"]}
    for key, value in results.items():
        if key == "summary":
            continue
        jsonable[key] = value.to_dict(orient="records") if value is not None else None
    return jsonable

All evals

In [ ]:
from huggingface_hub import HfApi
import json

cond = "condition-2-ur-5k"
EVAL_N_SAMPLES = None  # Set to an integer to limit rows per benchmark/language.

def main_english(batch_size: int = 32, n_samples=None):
    summary = {}
    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang="en", n_samples=n_samples, batch_size=batch_size)
    print("MGSM Done - EN")
    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang="en", n_samples=n_samples, batch_size=batch_size)
    print("XNLI Done - EN")
    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang="en", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang="en", n_samples=n_samples, batch_size=batch_size)
    print("CSQA Done - EN")
    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh, "mgsm_es": res_mgsm_es, "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh, "xnli_es": res_xnli_es, "xnli_ur": res_xnli_ur,
        "csqa_zh": res_csqa_zh, "csqa_es": res_csqa_es, "csqa_ur": res_csqa_ur,
    }

def main_language(batch_size: int = 32, n_samples=None):
    summary = {}
    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("MGSM Done - Lang")
    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("XNLI Done - Lang")
    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang="zh", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang="es", n_samples=n_samples, batch_size=batch_size)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang="ur", n_samples=n_samples, batch_size=batch_size)
    print("CSQA Done - Lang")
    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh, "mgsm_es": res_mgsm_es, "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh, "xnli_es": res_xnli_es, "xnli_ur": res_xnli_ur,
        "csqa_zh": res_csqa_zh, "csqa_es": res_csqa_es, "csqa_ur": res_csqa_ur,
    }

def save_results(results, summary_path, full_path):
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump({"summary": results["summary"]}, f, ensure_ascii=False, indent=2)

    json_results = {k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v)
                    for k, v in results.items()}
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    start = time.time()
    BATCH_SIZE = 256  
    print(f"Running evals with n_samples={EVAL_N_SAMPLES} and batch_size={BATCH_SIZE}")

    english_results  = main_english(batch_size=BATCH_SIZE, n_samples=EVAL_N_SAMPLES)
    language_results = main_language(batch_size=BATCH_SIZE, n_samples=EVAL_N_SAMPLES)
    english_summary_path  = f"/kaggle/working/{cond}_results_summary_english.json"
    english_full_path     = f"/kaggle/working/{cond}_results_english.json"
    language_summary_path = f"/kaggle/working/{cond}_results_summary_language.json"
    language_full_path    = f"/kaggle/working/{cond}_results_language.json"

    save_results(english_results,  english_summary_path,  english_full_path)
    save_results(language_results, language_summary_path, language_full_path)

    api = HfApi()
    for local_path, path_in_repo in [
        (english_full_path,  f"conditions/{cond}/results/english_prompt_results.json"),
        (language_full_path, f"conditions/{cond}/results/native_prompt_results.json"),
    ]:
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=path_in_repo,
            repo_id="legesher/language-decoded-experiments",
            repo_type="dataset",
        )
    end = time.time()
    print(f"Time taken: {elapsed} seconds")